In [1]:
import copy
from dataclasses import replace
from typing import List, Optional, Union

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Patch
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import NearestNeighbors
from umap import UMAP

from config import OUTPUTS_FOLDER
from config.experiment import ExperimentConfig
from config.task import generate_task_configs
from experiments import load_config
from utils.image_utils import load_adv_image
from utils.utils import get_device
from wrappers.cache import get_dataset, get_embedded_dataset
from wrappers.dataset import DatasetName
from wrappers.embedding import EmbedderName
from wrappers.vlm import VLMName

In [2]:
device = get_device(True)
mpl.use("pgf")

mpl.rcParams.update(
    {
        "pgf.texsystem": "pdflatex",  # or "xelatex"/"lualatex" if you prefer
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,
    }
)

In [3]:
def get_config(cfg_name: str) -> ExperimentConfig:
    exp_cfg = load_config(cfg_name)

    return ExperimentConfig(
        train=replace(exp_cfg.train, save_folder=exp_cfg.train.save_folder.parent / "paper"),
        eval=replace(exp_cfg.eval, results_folder=exp_cfg.eval.results_folder.parent / "paper"),
    )

## Jina UMAP visualization

In [12]:
# Load the model
ds = get_dataset(DatasetName.VIDORE_SYN_AI)

emb_ds = get_embedded_dataset(ds, EmbedderName.JINA_CLIP_2, False, False, device=device)

query_embeddings = np.asarray(emb_ds.query_embeddings.float().cpu())
image_embeddings = np.asarray(emb_ds.image_embeddings[:100, :].float().cpu())

# Permuting embeddings:
perm = np.random.permutation(len(query_embeddings))
query_embeddings = [query_embeddings[i] for i in perm]
image_embeddings = [image_embeddings[i] for i in perm]

X_jina = np.concatenate([image_embeddings, query_embeddings], axis=0)

In [14]:
# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,
    min_dist=0.01,
    metric="cosine",
    random_state=42,
)
X_embedded_jina = umap_model.fit_transform(X_jina)

# Plotting

labels = np.array([0] * len(query_embeddings) + [1] * len(image_embeddings))

colors = np.array(["red", "green"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_jina[:, 0],
    X_embedded_jina[:, 1],
    c=colors[labels],
    alpha=1,
    edgecolors="w",
    linewidth=0.5,
    s=100,
)

plt.tick_params(axis="both", which="major", labelsize=20)  # Set large font size for tick labels

legend_elements = [
    Patch(facecolor="red", edgecolor="w", label="Image Embeddings"),
    Patch(facecolor="green", edgecolor="w", label="Query Embeddings"),
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)

plt.title(f"UMAP Visualization of Jina-Clip Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}")

plt.grid(alpha=0.2)

metric = "cosine"
original_distances = squareform(pdist(X_jina, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_jina)
distances, indices = nn.kneighbors(X_jina)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_jina) / 2), len(X_jina)):
    my_list = list(indices[i][1:])
    # my_list.append(np.random.randint(0, len(X_jina)))
    for j in my_list:  # skip self (first neighbor)
        # j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0, 10) < 0:
            continue
        x1, y1 = X_embedded_jina[i]
        x2, y2 = X_embedded_jina[j]
        dist = original_distances[i, j]

        # Draw line between points
        # plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i - 100 in my_list:  # i-100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_jina[i]
        x2, y2 = X_embedded_jina[i - 100]
        dist = original_distances[i, i - 100]

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0, 10_000)
plt.savefig(OUTPUTS_FOLDER / f"jina_umap_noQuery_{rand_ind}.pgf")
plt.savefig(OUTPUTS_FOLDER / f"jina_umap_noQuery_{rand_ind}.pdf")
# plt.show()

/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Number of lines drawn for non-image neighbors: 100


## CLIP UMAP visualization

In [30]:
# Load the model
ds = get_dataset(DatasetName.VIDORE_SYN_AI)
emb_ds = get_embedded_dataset(ds, EmbedderName.CLIP_LARGE_PATCH14, False, False, device=device)

query_embeddings = np.asarray(emb_ds.query_embeddings.float().cpu())
image_embeddings = np.asarray(emb_ds.image_embeddings[:100, :].float().cpu())

clip_qwen_attack_img = None
clip_smolvlm_attack_img = None
exp_config = get_config("paper_non_targeted")
for task_config in generate_task_configs(exp_config):
    if task_config.model_name_embs[0] == EmbedderName.CLIP_LARGE_PATCH14:
        if task_config.vlm:
            if task_config.vlm.models[0] == VLMName.SMOLVLM_1_2B:
                clip_smolvlm_attack_img = load_adv_image(task_config, exp_config.train)
            elif task_config.vlm.models[0] == VLMName.QWEN_2p5_VL_3B:
                clip_qwen_attack_img = load_adv_image(task_config, exp_config.train)

emb_ds.add_adv_image(clip_qwen_attack_img)
image_embeddings = np.concatenate((image_embeddings, emb_ds.image_embeddings[-1:, :].float().cpu()))

# Permuting embeddings:
perm = np.random.permutation(len(query_embeddings))
query_embeddings = [query_embeddings[i] for i in perm]
image_embeddings[:100] = [image_embeddings[i] for i in perm]

X_clip = np.concatenate([image_embeddings, query_embeddings], axis=0)

In [34]:
# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,  # Controls local vs global structure balance
    min_dist=0.01,  # Minimum distance between embedded points
    metric="cosine",  # Distance metric (adjust based on data)
    random_state=42,
)
X_embedded_clip = umap_model.fit_transform(X_clip)

# Plotting
# Visualization with distinct colors

labels = np.array([0] * 100 + [1] * 100 + [2])

colors = np.array(["red", "green", "purple"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_clip[:, 0],
    X_embedded_clip[:, 1],
    c=colors[labels],
    alpha=1,
    edgecolors="w",
    linewidth=0.5,
    s=100,
)

plt.tick_params(axis="both", which="major", labelsize=20)  # Set large font size for tick labels

legend_elements = [
    Patch(facecolor="red", edgecolor="w", label="Image Embeddings"),
    Patch(facecolor="green", edgecolor="w", label="Query Embeddings"),
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)

plt.title(f"UMAP Visualization of Clip Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}")

plt.grid(alpha=0.2)

metric = "cosine"
original_distances = squareform(pdist(X_clip, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_clip)
distances, indices = nn.kneighbors(X_clip)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_clip) / 2), len(X_clip)):
    my_list = list(indices[i][1:])
    # my_list.append(np.random.randint(0, len(X_clip)))
    for j in my_list:  # skip self (first neighbor)
        # j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0, 10) < 0:
            continue
        x1, y1 = X_embedded_clip[i]
        x2, y2 = X_embedded_clip[j]
        dist = original_distances[i, j]

        # Draw line between points
        # plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i - 100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_clip[i]
        x2, y2 = X_embedded_clip[i - 100]
        dist = original_distances[i, i - 100]

        # Draw line between points
        # plt.plot([x1, x2], [y1, y2], 'red', linestyle='--', linewidth=0.2)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0, 10_000)
# plt.show()
plt.savefig(OUTPUTS_FOLDER / f"clip_umap_noQuery_{rand_ind}.pgf")
plt.savefig(OUTPUTS_FOLDER / f"clip_umap_noQuery_{rand_ind}.pdf")

/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due 

Number of lines drawn for non-image neighbors: 101


## ColPali UMAP Visualization

In [3]:
ds = get_dataset(DatasetName.VIDORE_SYN_AI)
emb_ds = get_embedded_dataset(ds, EmbedderName.COLPALI, False, False, device=device)

query_embeddings = list(emb_ds.query_embeddings.float().cpu())
image_embeddings = list(emb_ds.image_embeddings[:100, :].float().cpu())

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
# MaxSim score
def score_retrieval(
    query_embeddings: Union["torch.Tensor", List["torch.Tensor"]],
    passage_embeddings: Union["torch.Tensor", List["torch.Tensor"]],
    batch_size: int = 128,
    output_dtype: Optional["torch.dtype"] = None,
    output_device: Union["torch.device", str] = "cpu",
) -> "torch.Tensor":
    if len(query_embeddings) == 0:
        raise ValueError("No queries provided")
    if len(passage_embeddings) == 0:
        raise ValueError("No passages provided")

    if query_embeddings[0].device != passage_embeddings[0].device:
        raise ValueError("Queries and passages must be on the same device")

    if query_embeddings[0].dtype != passage_embeddings[0].dtype:
        raise ValueError("Queries and passages must have the same dtype")

    if output_dtype is None:
        output_dtype = query_embeddings[0].dtype

    scores: List[torch.Tensor] = []

    for i in range(0, len(query_embeddings), batch_size):
        batch_scores: List[torch.Tensor] = []
        batch_queries = torch.nn.utils.rnn.pad_sequence(
            query_embeddings[i : i + batch_size],
            batch_first=True,
            padding_value=0,
        )
        for j in range(0, len(passage_embeddings), batch_size):
            batch_passages = torch.nn.utils.rnn.pad_sequence(
                passage_embeddings[j : j + batch_size],
                batch_first=True,
                padding_value=0,
            )
            unnorm_score = torch.einsum("bnd,csd->bcns", batch_queries, batch_passages).max(dim=3)[0]
            print(f"unnorm_score: {unnorm_score.shape}")
            batch_scores.append(unnorm_score.sum(dim=2))
        scores.append(torch.cat(batch_scores, dim=1).to(output_dtype).to(output_device))

    return torch.cat(scores, dim=0)

In [11]:
# Permuting embeddings:
all_embeddings_norm = copy.deepcopy(query_embeddings) + copy.deepcopy(image_embeddings)
all_embeddings = copy.deepcopy(query_embeddings) + copy.deepcopy(image_embeddings)

for i in range(len(all_embeddings_norm)):
    all_embeddings_norm[i] = all_embeddings_norm[i] / all_embeddings_norm[i].shape[0]

# Calculate the scores
scores = score_retrieval(all_embeddings_norm, all_embeddings)
scores_sym = 1 - 0.5 * (scores + scores.T) + 1e-6
scores_sym = torch.clamp(scores_sym, min=0)

unnorm_score: torch.Size([128, 128, 1031])
unnorm_score: torch.Size([128, 72, 1031])
unnorm_score: torch.Size([72, 128, 1031])
unnorm_score: torch.Size([72, 72, 1031])


In [12]:
# Fit UMAP model using precomputed distances
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,  # Controls local vs global structure balance
    min_dist=0.01,  # Minimum distance between embedded points
    metric="precomputed",  # Distance metric (adjust based on data)
    # random_state=42
)
X_embedded_all = umap_model.fit_transform(scores_sym)

/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/cs/research/infosec/projects0/dristea/mumoRAG-attacks/.venv/lib/python3.11/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")


In [13]:
# Plotting

labels = np.array([0] * 100 + [1] * 100)

colors = np.array(["green", "red"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_all[:, 0],
    X_embedded_all[:, 1],
    c=colors[labels],
    alpha=1,
    edgecolors="w",
    linewidth=0.5,
    s=100,
)

legend_elements = [
    Patch(facecolor="red", edgecolor="w", label="Image Embeddings"),
    Patch(facecolor="green", edgecolor="w", label="Query Embeddings"),
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)

n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric="precomputed")  # +1 because point is neighbor to itself
nn.fit(scores_sym)
distances, indices = nn.kneighbors(scores_sym)

# Draw lines and annotate distances
nn_not_image_counter = 0
for i in range(int(len(all_embeddings) / 2)):
    my_list = list(indices[i][1:])
    if len(my_list) == 0:
        print(i)
    if np.random.randint(0, 10) < 0:
        my_list.append(np.random.randint(0, len(all_embeddings)))
    for point_ind, j in enumerate(my_list):  # skip self (first neighbor)
        # j = np.random.randint(0, len(cos_data))
        # if i == j or np.random.randint(0,10) < 0:
        #    continue
        x1, y1 = X_embedded_all[i]
        x2, y2 = X_embedded_all[j]
        dist = scores_sym[i, j]

        if point_ind > 0:
            line_color = "red"
            text_color = "red"
        else:
            line_color = "gray"
            text_color = "black"
        # Draw line between points
        # plt.plot([x1, x2], [y1, y2], line_color, linestyle='--', linewidth=0.5)

        if np.linalg.norm(np.array([x1, y1]) - np.array([x2, y2]), ord=2) < 0.5:
            step = 0.5
            eps_x = np.random.uniform(-step, step)
            eps_y = np.random.uniform(-step, step)
        else:
            eps_x = 0
            eps_y = 0
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2 + eps_x, (y1 + y2) / 2 + eps_y
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color=text_color, ha='center')

    if not i + 100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_all[i]
        x2, y2 = X_embedded_all[i + 100]
        dist = scores_sym[i, i + 100]

        # Draw line between points
        plt.plot([x1, x2], [y1, y2], "blue", linestyle="--", linewidth=0.5)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

plt.title(f"UMAP Visualization of ColPali Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}")
# plt.show()
rand_ind = np.random.randint(0, 10_000)

plt.savefig(OUTPUTS_FOLDER / f"colpali_umap_noQuery_{rand_ind}.pgf")
plt.savefig(OUTPUTS_FOLDER / f"colpali_umap_noQuery_{rand_ind}.pdf")
print(f"pdf saved with id: {rand_ind}")
print(f"Number of queries that do not have the corresponding image in the nearest neighbors: {nn_not_image_counter}")

pdf saved with id: 7036
Number of queries that do not have the corresponding image in the nearest neighbors: 43


## gme-Qwen2 UMAP Visualization

In [ ]:
ds = get_dataset(DatasetName.VIDORE_SYN_AI)
emb_ds = get_embedded_dataset(ds, EmbedderName.QWEN2_GME_2B, False, False, device=device)

query_embeddings = list(emb_ds.query_embeddings.float().cpu())
image_embeddings = list(emb_ds.image_embeddings[:100, :].float().cpu())

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# Prepare the data for UMAP
query_embeddings = np.array(query_embeddings)
image_embeddings = np.array(image_embeddings)

X_qwen = np.concatenate([query_embeddings, image_embeddings], axis=0)
X_qwen = X_qwen[:, 0, :]

# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,  # Controls local vs global structure balance
    min_dist=0.01,  # Minimum distance between embedded points
    metric="cosine",  # Distance metric (adjust based on data)
    random_state=42,
)
X_embedded_qwen = umap_model.fit_transform(X_qwen)

In [ ]:
# Plotting
labels = np.array([0] * len(image_embeddings) + [1] * len(query_embeddings))

colors = np.array(["green", "red"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_qwen[:, 0],
    X_embedded_qwen[:, 1],
    c=colors[labels],
    alpha=1,
    edgecolors="w",
    linewidth=0.5,
    s=100,
)

plt.tick_params(axis="both", which="major", labelsize=20)  # Set large font size for tick labels

legend_elements = [
    Patch(facecolor="red", edgecolor="w", label="Image Embeddings"),
    Patch(facecolor="green", edgecolor="w", label="Query Embeddings"),
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)

plt.title(f"UMAP Visualization of Qwen Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}")

plt.grid(alpha=0.2)

metric = "cosine"
original_distances = squareform(pdist(X_qwen, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_qwen)
distances, indices = nn.kneighbors(X_qwen)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_qwen) / 2)):
    my_list = list(indices[i][1:])
    # my_list.append(np.random.randint(0, len(X_qwen)))
    for j in my_list:  # skip self (first neighbor)
        # j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0, 10) < 0:
            continue
        x1, y1 = X_embedded_qwen[i]
        x2, y2 = X_embedded_qwen[j]
        dist = original_distances[i, j]

        # Draw line between points
        # plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i + 100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_qwen[i]
        x2, y2 = X_embedded_qwen[i + 100]
        dist = original_distances[i, i + 100]

        # Draw line between points
        plt.plot([x1, x2], [y1, y2], "blue", linestyle="--", linewidth=0.2)

        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        # plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0, 10_000)
# plt.show()
plt.savefig(OUTPUTS_FOLDER / f"qwen_umap_noQuery_{rand_ind}.pgf")
plt.savefig(OUTPUTS_FOLDER / f"qwen_umap_noQuery_{rand_ind}.pdf")